In [ ]:
import os
import re
import sys
import shutil
from pathlib import Path
from collections import defaultdict
import time
import numpy as np
from osgeo import gdal
import os

In [ ]:
# USER SETTINGS

# Text file containing all downloaded ESACCI .nc paths from 2000–2025.
FILE_LIST = Path("/mnt/bender/gen_cog/snow_cover_dlr/test_esacci.txt")

OUTPUT_DIR = Path("/mnt/bender/gen_cog/snow_cover_dlr/esacci_snow_monthly")
TMP_DIR = Path("/mnt/bender/gen_cog/snow_cover_dlr/tmp_esacci_snow_monthly")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# Processing period
START_YEAR = 2000
END_YEAR = 2001

# Process all available months between START_YEAR and END_YEAR.
PROCESS_ALL_MONTHS = False

# If PROCESS_ALL_MONTHS = False, only these months are processed.
# Format: YYYYMM
month = "01"

SELECTED_MONTHS = [
    f"{year}{month}"
    for year in range(2001, END_YEAR+1)
]

GDAL_THREADS = "8"
GDAL_CACHE_MB = "65536"  # 64 GB GDAL cache

# Output options
INCLUDE_COUNTS = False
WRITE_COG = False
OVERWRITE = True

# Data settings
VALID_MIN = 0
VALID_MAX = 100

OUT_NODATA = 255
SRC_NODATA = 255

# Special ESACCI class: permanent snow and ice. Treat this as valid and equivalent to 100% snow cover.
PERMANENT_SNOW_ICE_VALUE = 215
PERMANENT_SNOW_ICE_AS_SCFG = 100

# scf = days(SCFG >= 10) / count_valid * 100
SNOW_THRESH = 10

# 1 / 240 degree 500 m-like WGS84 resolution
TARGET_RES = 0.004166666666666666609

VERSION = "v20260610"

CANDIDATE_VARS = [
    "scfg",
    "SCFG",
    "snow_cover_fraction",
    "snow_fraction",
]

In [ ]:
# FIX PROJ PATH BEFORE IMPORTING GDAL
candidate_proj_dirs = [
    Path(os.environ.get("CONDA_PREFIX", "")) / "share" / "proj",
    Path(sys.prefix) / "share" / "proj",
    Path("/opt/conda/share/proj"),
    Path("/usr/share/proj"),
    Path("/usr/local/share/proj"),
]

for proj_dir in candidate_proj_dirs:
    if (proj_dir / "proj.db").exists():
        os.environ["PROJ_DATA"] = str(proj_dir)
        os.environ["PROJ_LIB"] = str(proj_dir)
        break

from osgeo import gdal, osr

gdal.UseExceptions()

if "PROJ_DATA" in os.environ:
    gdal.SetConfigOption("PROJ_DATA", os.environ["PROJ_DATA"])
    gdal.SetConfigOption("PROJ_LIB", os.environ["PROJ_LIB"])

gdal.SetConfigOption("GDAL_NUM_THREADS", GDAL_THREADS)
gdal.SetConfigOption("GDAL_CACHEMAX", GDAL_CACHE_MB)
gdal.SetConfigOption("VSI_CACHE", "TRUE")
gdal.SetConfigOption("VSI_CACHE_SIZE", str(1024 * 1024 * 1024))

In [ ]:
# HELPER FUNCTIONS

def extract_date(path):
    match = re.match(r"(\d{8})", path.name)

    if match is None:
        raise ValueError("Cannot extract date from filename: {}".format(path.name))

    return match.group(1)


def extract_yyyymm(path):
    return extract_date(path)[:6]


def read_nc_file_list(txt_path):
    if not txt_path.exists():
        raise FileNotFoundError("File list does not exist: {}".format(txt_path))

    nc_paths = []

    with txt_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if not line.endswith(".nc"):
                continue

            path = Path(line)

            try:
                date = extract_date(path)
            except ValueError:
                continue

            year = int(date[:4])

            if START_YEAR <= year <= END_YEAR:
                nc_paths.append(path)

    if not nc_paths:
        raise ValueError(
            "No .nc files found between {} and {} in file list: {}"
            .format(START_YEAR, END_YEAR, txt_path)
        )

    return sorted(nc_paths)


def group_files_by_month(nc_paths):

    files_by_month = defaultdict(list)

    for path in nc_paths:
        yyyymm = extract_yyyymm(path)
        files_by_month[yyyymm].append(path)

    grouped = {}

    for month in sorted(files_by_month.keys()):
        grouped[month] = sorted(files_by_month[month])

    return grouped


def choose_months(files_by_month):

    if PROCESS_ALL_MONTHS:
        return sorted(files_by_month.keys())

    missing = []

    for month in SELECTED_MONTHS:
        if month not in files_by_month:
            missing.append(month)

    if missing:
        raise ValueError("Requested months not found in file list: {}".format(missing))

    return SELECTED_MONTHS

In [ ]:
def find_snow_subdataset(nc_path):

    root_ds = gdal.Open(str(nc_path), gdal.GA_ReadOnly)

    if root_ds is None:
        raise RuntimeError("Could not open NetCDF: {}".format(nc_path))

    subdatasets = root_ds.GetSubDatasets()

    if not subdatasets:
        root_ds = None
        return str(nc_path)

    selected_sds = None

    for sds_name, sds_desc in subdatasets:
        sds_name_l = sds_name.lower()
        sds_desc_l = sds_desc.lower()

        for var in CANDIDATE_VARS:
            var_l = var.lower()

            if ":{}".format(var_l) in sds_name_l or var_l in sds_desc_l:
                selected_sds = sds_name
                break

        if selected_sds is not None:
            break

    if selected_sds is None:
        print("\nAvailable subdatasets in {}:".format(nc_path.name), flush=True)

        for sds_name, sds_desc in subdatasets:
            print("  {}".format(sds_name), flush=True)
            print("  {}".format(sds_desc), flush=True)

        root_ds = None

        raise ValueError(
            "Could not find SCFG snow-cover variable in {}. "
            "Add the correct variable name to CANDIDATE_VARS.".format(nc_path.name)
        )

    root_ds = None
    return selected_sds


def open_scfg_dataset(nc_path):

    sds_name = find_snow_subdataset(nc_path)

    ds = gdal.Open(sds_name, gdal.GA_ReadOnly)

    if ds is None:
        raise RuntimeError("Could not open SCFG subdataset: {}".format(sds_name))

    return ds


def get_bounds_from_dataset(ds):
    
    gt = ds.GetGeoTransform()

    left = gt[0]
    top = gt[3]
    right = left + gt[1] * ds.RasterXSize
    bottom = top + gt[5] * ds.RasterYSize

    return left, bottom, right, top


def build_native_grid(ref_ds):

    left, bottom, right, top = get_bounds_from_dataset(ref_ds)

    projection = ref_ds.GetProjection()

    if not projection:
        srs = osr.SpatialReference()
        srs.ImportFromEPSG(4326)
        projection = srs.ExportToWkt()

    return {
        "left": left,
        "bottom": bottom,
        "right": right,
        "top": top,
        "x_size": ref_ds.RasterXSize,
        "y_size": ref_ds.RasterYSize,
        "geo_transform": ref_ds.GetGeoTransform(),
        "projection": projection,
    }


def build_target_grid(ref_ds):

    left, bottom, right, top = get_bounds_from_dataset(ref_ds)

    x_size = int(round((right - left) / TARGET_RES))
    y_size = int(round((top - bottom) / TARGET_RES))

    target_gt = (
        left,
        TARGET_RES,
        0.0,
        top,
        0.0,
        -TARGET_RES,
    )

    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    target_wkt = srs.ExportToWkt()

    return {
        "left": left,
        "bottom": bottom,
        "right": right,
        "top": top,
        "x_size": x_size,
        "y_size": y_size,
        "geo_transform": target_gt,
        "projection": target_wkt,
    }


def print_memory_estimate(grid, n_days, yyyymm):
    n_pixels = grid["x_size"] * grid["y_size"]

    gb_uint8 = n_pixels / 1024**3
    gb_float32 = n_pixels * 4 / 1024**3

    print("[{}] Memory estimate:".format(yyyymm), flush=True)
    print("[{}]   Pixels:           {:,}".format(yyyymm, n_pixels), flush=True)
    print("[{}]   One uint8 array:   {:.2f} GB".format(yyyymm, gb_uint8), flush=True)
    print("[{}]   One float32 array: {:.2f} GB".format(yyyymm, gb_float32), flush=True)
    print("[{}]   Daily files:       {}".format(yyyymm, n_days), flush=True)


def read_daily_native(src_ds):

    arr = src_ds.GetRasterBand(1).ReadAsArray()

    if arr is None:
        raise RuntimeError("Could not read native daily array.")

    return arr.astype(np.float32, copy=False)

In [ ]:
def make_output_name(param_code, start_date, end_date):
    """
    ESACCI naming convention.
    """
    return (
        "snowcover_esacci_{}_500m_s_{}_{}_go_epsg.4326_{}.tif"
        .format(param_code, start_date, end_date, VERSION)
    )


def create_output_geotiff(path, grid):
    
    #Create writable uint8 tiled GeoTIFF. Used for temporary native-grid monthly layers.
    
    if path.exists() and OVERWRITE:
        path.unlink()

    driver = gdal.GetDriverByName("GTiff")

    ds_out = driver.Create(
        str(path),
        grid["x_size"],
        grid["y_size"],
        1,
        gdal.GDT_Byte,
        options=[
            "TILED=YES",
            "BLOCKXSIZE=2048",
            "BLOCKYSIZE=2048",
            "COMPRESS=DEFLATE",
            "BIGTIFF=IF_SAFER",
            "NUM_THREADS={}".format(GDAL_THREADS),
        ],
    )

    if ds_out is None:
        raise RuntimeError("Could not create GeoTIFF: {}".format(path))

    ds_out.SetGeoTransform(grid["geo_transform"])
    ds_out.SetProjection(grid["projection"])

    band = ds_out.GetRasterBand(1)
    band.SetNoDataValue(OUT_NODATA)

    return ds_out


def resample_monthly_to_final(tmp_tif, final_tif, target_grid, resample_alg):
    
    #Resample one finished monthly native-grid layer to the final WGS84 grid. The final output is written as COG if WRITE_COG=True, otherwise as GeoTIFF.
    
    if final_tif.exists() and OVERWRITE:
        final_tif.unlink()

    if WRITE_COG:
        out_format = "COG"
        creation_options = [
            "COMPRESS=DEFLATE",
            "BLOCKSIZE=2048",
            "BIGTIFF=IF_SAFER",
            "OVERVIEWS=AUTO",
            "NUM_THREADS={}".format(GDAL_THREADS),
        ]
    else:
        out_format = "GTiff"
        creation_options = [
            "TILED=YES",
            "BLOCKXSIZE=2048",
            "BLOCKYSIZE=2048",
            "COMPRESS=DEFLATE",
            "BIGTIFF=IF_SAFER",
            "NUM_THREADS={}".format(GDAL_THREADS),
        ]

    result = gdal.Warp(
        str(final_tif),
        str(tmp_tif),
        options=gdal.WarpOptions(
            format=out_format,
            dstSRS="EPSG:4326",
            outputBounds=(
                target_grid["left"],
                target_grid["bottom"],
                target_grid["right"],
                target_grid["top"],
            ),
            width=target_grid["x_size"],
            height=target_grid["y_size"],
            resampleAlg=resample_alg,
            outputType=gdal.GDT_Byte,
            srcNodata=OUT_NODATA,
            dstNodata=OUT_NODATA,
            multithread=True,
            creationOptions=creation_options,
            warpOptions=[
                "NUM_THREADS={}".format(GDAL_THREADS),
            ],
        ),
    )

    if result is None:
        raise RuntimeError("Failed to create final GeoTIFF: {}".format(final_tif))

    result = None


def make_output_paths(start_date, end_date):
    """
    p   = monthly mean daily SCFG
    scf = monthly snow-cover frequency based on SCFG >= 10
    cv  = count valid
    ci  = count invalid
    """
    output_paths = {
        "p": OUTPUT_DIR / make_output_name("p", start_date, end_date),
        "scf": OUTPUT_DIR / make_output_name("scf", start_date, end_date),
    }

    if INCLUDE_COUNTS:
        output_paths["cv"] = OUTPUT_DIR / make_output_name("cv", start_date, end_date)
        output_paths["ci"] = OUTPUT_DIR / make_output_name("ci", start_date, end_date)

    return output_paths


def make_working_paths(yyyymm, final_paths):
    
    #Always write temporary native-grid monthly GeoTIFFs first. These are then resampled once to the final grid.
    
    month_tmp_dir = TMP_DIR / yyyymm

    if month_tmp_dir.exists():
        shutil.rmtree(month_tmp_dir)

    month_tmp_dir.mkdir(parents=True, exist_ok=True)

    working_paths = {}

    for code, final_path in final_paths.items():
        working_paths[code] = month_tmp_dir / final_path.name.replace(".tif", "_native_tmp.tif")

    return working_paths


def write_final_outputs(yyyymm, working_paths, final_paths, target_grid):
    """
    Resample finished monthly native-grid GeoTIFFs to the final target grid.
    p and scf are continuous percentage-like layers, so cubic spline is used.
    cv and ci are count layers, so nearest-neighbour is used.
    """
    for code, tmp_path in working_paths.items():
        final_path = final_paths[code]

        if code in ["cv", "ci"]:
            resample_alg = gdal.GRA_NearestNeighbour
        else:
            resample_alg = gdal.GRA_CubicSpline

        print("[{}] Resampling monthly {} to final output: {}".format(
            yyyymm,
            code,
            final_path,
        ), flush=True)

        resample_monthly_to_final(
            tmp_tif=tmp_path,
            final_tif=final_path,
            target_grid=target_grid,
            resample_alg=resample_alg,
        )

    month_tmp_dir = TMP_DIR / yyyymm

    if month_tmp_dir.exists():
        shutil.rmtree(month_tmp_dir)


def write_array_to_geotiff(path, grid, array):
    ds = create_output_geotiff(path, grid)
    ds.GetRasterBand(1).WriteArray(array)
    ds.FlushCache()
    ds = None

In [ ]:
def format_elapsed(seconds):
    
    #Format elapsed time as seconds, minutes, or hours.
    
    if seconds < 60:
        return "{:.2f} s".format(seconds)

    minutes, seconds = divmod(seconds, 60)

    if minutes < 60:
        return "{:.0f} min {:.2f} s".format(minutes, seconds)

    hours, minutes = divmod(minutes, 60)

    return "{:.0f} h {:.0f} min {:.2f} s".format(hours, minutes, seconds)


def print_step_time(yyyymm, label, start_time):
    elapsed = time.perf_counter() - start_time

    print(
        "[{}] Time - {}: {}".format(
            yyyymm,
            label,
            format_elapsed(elapsed),
        ),
        flush=True,
    )

    return elapsed

In [ ]:
def process_month(yyyymm, month_files):
    """
    Process one month using all GDAL threads.
    Monthly analysis is done in native resolution.
    Only the final monthly layers are resampled to the target grid.

    Outputs:
    p   = monthly mean of daily SCFG values
    scf = percentage of valid days with SCFG >= 10
    cv  = count valid days
    ci  = count invalid days

    Special handling:
    SCFG value 215 is permanent snow/ice. It is counted as valid and treated as 100% snow cover.
    """
    month_start_time = time.perf_counter()

    print("\n" + "=" * 80, flush=True)
    print("[{}] Processing month".format(yyyymm), flush=True)
    print("[{}] Daily NetCDF files: {}".format(yyyymm, len(month_files)), flush=True)
    print("[{}] GDAL threads: {}".format(yyyymm, GDAL_THREADS), flush=True)
    print("=" * 80, flush=True)

    step_start_time = time.perf_counter()

    dates = [extract_date(path) for path in month_files]
    start_date = min(dates)
    end_date = max(dates)

    final_paths = make_output_paths(start_date, end_date)

    print_step_time(yyyymm, "prepare month paths", step_start_time)

    if not OVERWRITE:
        step_start_time = time.perf_counter()

        all_exist = all(path.exists() for path in final_paths.values())

        print_step_time(yyyymm, "check existing outputs", step_start_time)

        if all_exist:
            print("[{}] Skipped; all outputs already exist.".format(yyyymm), flush=True)

            month_elapsed = time.perf_counter() - month_start_time
            print("[{}] Month total time: {}".format(
                yyyymm,
                format_elapsed(month_elapsed),
            ), flush=True)

            return

    print("[{}] Opening reference NetCDF...".format(yyyymm), flush=True)

    step_start_time = time.perf_counter()

    ref_ds = open_scfg_dataset(month_files[0])

    native_grid = build_native_grid(ref_ds)
    target_grid = build_target_grid(ref_ds)

    ref_ds = None

    print_step_time(yyyymm, "open reference and build grids", step_start_time)

    print("[{}] Native grid:".format(yyyymm), flush=True)
    print("[{}]   Width:      {}".format(yyyymm, native_grid["x_size"]), flush=True)
    print("[{}]   Height:     {}".format(yyyymm, native_grid["y_size"]), flush=True)
    print(
        "[{}]   Bounds:     {}, {}, {}, {}"
        .format(
            yyyymm,
            native_grid["left"],
            native_grid["bottom"],
            native_grid["right"],
            native_grid["top"],
        ),
        flush=True,
    )

    print("[{}] Final target grid:".format(yyyymm), flush=True)
    print("[{}]   Resolution: {} degree".format(yyyymm, TARGET_RES), flush=True)
    print("[{}]   Width:      {}".format(yyyymm, target_grid["x_size"]), flush=True)
    print("[{}]   Height:     {}".format(yyyymm, target_grid["y_size"]), flush=True)
    print(
        "[{}]   Bounds:     {}, {}, {}, {}"
        .format(
            yyyymm,
            target_grid["left"],
            target_grid["bottom"],
            target_grid["right"],
            target_grid["top"],
        ),
        flush=True,
    )

    print_memory_estimate(native_grid, len(month_files), yyyymm)

    step_start_time = time.perf_counter()

    working_paths = make_working_paths(yyyymm, final_paths)

    print_step_time(yyyymm, "prepare temporary output paths", step_start_time)

    shape = (native_grid["y_size"], native_grid["x_size"])

    print("[{}] Allocating native-grid monthly arrays...".format(yyyymm), flush=True)

    step_start_time = time.perf_counter()

    count_valid = np.zeros(shape, dtype=np.uint8)
    count_snowy = np.zeros(shape, dtype=np.uint8)
    sum_scfg = np.zeros(shape, dtype=np.float32)

    print_step_time(yyyymm, "allocate monthly arrays", step_start_time)

    print("[{}] Processing daily rasters...".format(
        yyyymm
    ), flush=True)

    daily_total_start_time = time.perf_counter()

    for i, nc_path in enumerate(month_files, start=1):
        daily_start_time = time.perf_counter()

        print(
            "[{}]   Reading daily raster {}/{}: {}"
            .format(yyyymm, i, len(month_files), nc_path.name),
            flush=True,
        )

        src_ds = open_scfg_dataset(nc_path)
        arr = read_daily_native(src_ds)
        src_ds = None

        # Normal SCFG values are valid from 0–100.
        valid_scfg = (
            np.isfinite(arr)
            & (arr >= VALID_MIN)
            & (arr <= VALID_MAX)
        )

        # ESACCI special class: permanent snow and ice.
        # Treat it as valid and as 100% snow cover.
        permanent_snow_ice = arr == PERMANENT_SNOW_ICE_VALUE

        valid = valid_scfg | permanent_snow_ice

        count_valid += valid.astype(np.uint8)

        snowy = (
            valid_scfg & (arr >= SNOW_THRESH)
        ) | permanent_snow_ice

        count_snowy += snowy.astype(np.uint8)

        # Add normal 0–100 SCFG values.
        # Do not add raw 215 values to the monthly mean.
        arr[~valid_scfg] = 0.0
        sum_scfg += arr

        # Add permanent snow/ice as 100% SCFG.
        sum_scfg += (
            permanent_snow_ice.astype(np.float32)
            * PERMANENT_SNOW_ICE_AS_SCFG
        )

        del arr
        del valid_scfg
        del permanent_snow_ice
        del valid
        del snowy

        daily_elapsed = time.perf_counter() - daily_start_time

        print(
            "[{}]   Daily raster {}/{} time: {}".format(
                yyyymm,
                i,
                len(month_files),
                format_elapsed(daily_elapsed),
            ),
            flush=True,
        )

    print_step_time(yyyymm, "process all daily rasters", daily_total_start_time)

    print("[{}] Creating monthly p layer in native resolution...".format(
        yyyymm
    ), flush=True)

    step_start_time = time.perf_counter()

    p = np.full(shape, OUT_NODATA, dtype=np.uint8)
    valid_pixels = count_valid > 0

    # p = monthly mean of daily SCFG values
    np.divide(
        sum_scfg,
        count_valid,
        out=sum_scfg,
        where=valid_pixels,
    )

    np.rint(sum_scfg, out=sum_scfg)
    np.clip(sum_scfg, 0, 100, out=sum_scfg)

    p[valid_pixels] = sum_scfg[valid_pixels].astype(np.uint8)

    del valid_pixels

    print_step_time(yyyymm, "create monthly p layer", step_start_time)

    print("[{}] Creating monthly scf layer in native resolution...".format(
        yyyymm
    ), flush=True)

    step_start_time = time.perf_counter()

    scf = np.full(shape, OUT_NODATA, dtype=np.uint8)
    valid_pixels = count_valid > 0

    # Reuse sum_scfg as temporary float32 array.
    sum_scfg.fill(0.0)
    sum_scfg += count_snowy
    sum_scfg *= 100.0

    # scf = count_snowy / count_valid * 100
    np.divide(
        sum_scfg,
        count_valid,
        out=sum_scfg,
        where=valid_pixels,
    )

    np.rint(sum_scfg, out=sum_scfg)
    np.clip(sum_scfg, 0, 100, out=sum_scfg)

    scf[valid_pixels] = sum_scfg[valid_pixels].astype(np.uint8)

    del valid_pixels
    del sum_scfg

    print_step_time(yyyymm, "create monthly scf layer", step_start_time)

    print("[{}] Writing temporary native-grid GeoTIFF arrays...".format(
        yyyymm
    ), flush=True)

    temp_write_start_time = time.perf_counter()

    step_start_time = time.perf_counter()
    write_array_to_geotiff(working_paths["p"], native_grid, p)
    print_step_time(yyyymm, "write temporary p GeoTIFF", step_start_time)
    del p

    step_start_time = time.perf_counter()
    write_array_to_geotiff(working_paths["scf"], native_grid, scf)
    print_step_time(yyyymm, "write temporary scf GeoTIFF", step_start_time)
    del scf

    if INCLUDE_COUNTS:
        step_start_time = time.perf_counter()
        write_array_to_geotiff(working_paths["cv"], native_grid, count_valid)
        print_step_time(yyyymm, "write temporary cv GeoTIFF", step_start_time)

        step_start_time = time.perf_counter()
        count_invalid = (len(month_files) - count_valid).astype(np.uint8)
        write_array_to_geotiff(working_paths["ci"], native_grid, count_invalid)
        print_step_time(yyyymm, "create and write temporary ci GeoTIFF", step_start_time)

        del count_invalid

    print_step_time(yyyymm, "write all temporary native GeoTIFFs", temp_write_start_time)

    del count_valid
    del count_snowy

    final_write_start_time = time.perf_counter()

    write_final_outputs(
        yyyymm=yyyymm,
        working_paths=working_paths,
        final_paths=final_paths,
        target_grid=target_grid,
    )

    print_step_time(yyyymm, "resample and write final outputs", final_write_start_time)

    month_elapsed = time.perf_counter() - month_start_time

    print("[{}] Finished".format(yyyymm), flush=True)
    print("[{}] Month total time: {}".format(
        yyyymm,
        format_elapsed(month_elapsed),
    ), flush=True)

    print("[{}] Outputs:".format(yyyymm), flush=True)

    for code, path in final_paths.items():
        print("[{}]   {}: {}".format(yyyymm, code, path), flush=True)

In [ ]:
def main():
    print("Reading input file list...", flush=True)
    nc_paths = read_nc_file_list(FILE_LIST)

    print("Total NetCDF files found between {} and {}: {}".format(
        START_YEAR,
        END_YEAR,
        len(nc_paths),
    ), flush=True)

    files_by_month = group_files_by_month(nc_paths)

    print("Months found:", flush=True)

    for month, paths in files_by_month.items():
        print("  {}: {} files".format(month, len(paths)), flush=True)

    months_to_process = choose_months(files_by_month)

    print("Processing mode:", flush=True)
    print("  JupyterLab native", flush=True)
    print("  One month at a time", flush=True)
    print("  Daily rasters processed in native resolution", flush=True)
    print("  Permanent snow/ice value 215 treated as 100% snow cover", flush=True)
    print("  Monthly layers resampled once to final target grid", flush=True)
    print("  Years: {}–{}".format(START_YEAR, END_YEAR), flush=True)
    print("  GDAL threads per month: {}".format(GDAL_THREADS), flush=True)
    print("Months selected:", flush=True)

    for month in months_to_process:
        print("  {}".format(month), flush=True)

    for month in months_to_process:
        process_month(month, files_by_month[month])

    print("\nAll selected months completed successfully.", flush=True)

In [ ]:
main()